GLOBAL IMPORTS

In [10]:
import os
import sys
import glob
import json
import csv
import time

from pathlib import Path

import cv2
import numpy as np
import pyrealsense2 as rs
import onnxruntime as ort


BAG → RGB + DEPTH EXTRACTION

In [11]:
# =====================================================
# CHANGE THIS ONLY
# =====================================================
FFB_ID = 17   # ← change to 10, 11, 12, etc.

# =====================================================
# PATHS (AUTO-UPDATED)
# =====================================================
bag_path = rf"C:\Users\admin\Documents\Vicki\School stuff\SEGP\FFB Samples\FFB{FFB_ID}\ffb{FFB_ID}Depth_3D.bag"
out_dir  = rf"sample_{FFB_ID:03d}"
os.makedirs(out_dir, exist_ok=True)

print(f"Processing FFB {FFB_ID}")
print("Bag file:", bag_path)
print("Output dir:", out_dir)

# =====================================================
# REALSENSE PIPELINE
# =====================================================
pipe = rs.pipeline()
cfg = rs.config()
cfg.enable_device_from_file(bag_path, repeat_playback=False)
cfg.enable_stream(rs.stream.color)
cfg.enable_stream(rs.stream.depth)

profile = pipe.start(cfg)

align = rs.align(rs.stream.color)
depth_sensor = profile.get_device().first_depth_sensor()
depth_scale = depth_sensor.get_depth_scale()

color_stream = profile.get_stream(rs.stream.color).as_video_stream_profile()
intr = color_stream.get_intrinsics()

# =====================================================
# SAVE INTRINSICS
# =====================================================
intr_path = os.path.join(out_dir, "intrinsics.json")
with open(intr_path, "w") as f:
    json.dump({
        "fx": intr.fx,
        "fy": intr.fy,
        "cx": intr.ppx,
        "cy": intr.ppy,
        "width": intr.width,
        "height": intr.height,
        "depth_scale": depth_scale
    }, f, indent=2)

print("Saved intrinsics to:", intr_path)

# =====================================================
# EXTRACT FRAMES
# =====================================================
i = 0
try:
    while True:
        frames = pipe.wait_for_frames()
        frames = align.process(frames)

        depth = frames.get_depth_frame()
        color = frames.get_color_frame()
        if not depth or not color:
            continue

        i += 1
        rgb = np.asanyarray(color.get_data())
        dep = np.asanyarray(depth.get_data())

        cv2.imwrite(os.path.join(out_dir, f"rgb_{i:04d}.png"),
                    cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
        cv2.imwrite(os.path.join(out_dir, f"depth_{i:04d}.png"), dep)

except Exception as e:
    print("Stopped:", e)

finally:
    pipe.stop()
    print(f"Done. Extracted {i} frames.")


Processing FFB 17
Bag file: C:\Users\admin\Documents\Vicki\School stuff\SEGP\FFB Samples\FFB17\ffb17Depth_3D.bag
Output dir: sample_017
Saved intrinsics to: sample_017\intrinsics.json
Stopped: Frame didn't arrive within 5000
Done. Extracted 79 frames.


DEPTH VALIDITY CHECK

In [12]:
FFB_ID = 17
frames_dir = f"sample_{FFB_ID:03d}"

with open(f"{frames_dir}/intrinsics.json", "r") as f:
    meta = json.load(f)

scale = meta["depth_scale"]

depth_files = sorted(glob.glob(f"{frames_dir}/depth_*.png"))
assert len(depth_files) > 0

D = cv2.imread(depth_files[0], cv2.IMREAD_UNCHANGED)
assert D is not None

valid = D > 0

print(f"FFB {FFB_ID}")
print("depth_scale:", scale)
print("valid %:", 100 * valid.mean())

vals_m = (D[valid] * scale).astype(float)
print(
    "valid depth (m): min",
    vals_m.min(),
    "med",
    np.median(vals_m),
    "max",
    vals_m.max()
)


FFB 17
depth_scale: 0.0010000000474974513
valid %: 92.78385416666667
valid depth (m): min 0.4480000212788582 med 1.5480000735260546 max 1.6790000797482207


FRAME DISCOVERY

In [13]:
FFB_ID = 17
frames_dir = f"sample_{FFB_ID:03d}"

print("cwd:", os.getcwd())
print(f"{frames_dir} exists?", os.path.isdir(frames_dir))

rgb_files = sorted(glob.glob(f"{frames_dir}/rgb_*.png"))
print("rgb files found:", len(rgb_files))
print("first 3:", rgb_files[:3])


cwd: C:\Users\admin\Documents\Vicki\School stuff\SEGP
sample_017 exists? True
rgb files found: 79
first 3: ['sample_017\\rgb_0001.png', 'sample_017\\rgb_0002.png', 'sample_017\\rgb_0003.png']


DEPENDENCY CHECK

In [14]:
def ensure(pkg):
    try:
        __import__(pkg)
        print(f"[OK] {pkg} already installed")
    except ImportError:
        print(f"[INSTALLING] {pkg}")
        !{sys.executable} -m pip install {pkg}

ensure("onnxruntime")
ensure("cv2")
ensure("numpy")

print("ONNX Runtime version:", ort.__version__)
print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)


[OK] onnxruntime already installed
[OK] cv2 already installed
[OK] numpy already installed
ONNX Runtime version: 1.23.2
OpenCV version: 4.10.0
NumPy version: 2.2.6


LOCAL YOLO DETECTOR CLASS

In [15]:
class LocalYOLODetector:
    def __init__(self, model_path, input_size=640, conf_threshold=0.25):
        self.input_size = input_size
        self.conf_threshold = conf_threshold

        model_path = Path(model_path)
        if not model_path.exists():
            raise FileNotFoundError(f"ONNX model not found: {model_path}")

        self.session = ort.InferenceSession(str(model_path))
        self.input_name = self.session.get_inputs()[0].name
        self.output_name = self.session.get_outputs()[0].name

        print(f"[OK] Loaded ONNX YOLO model: {model_path}")

    def _preprocess(self, image):
        img = cv2.resize(image, (self.input_size, self.input_size))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))
        return np.expand_dims(img, axis=0)

    def detect(self, image):
        H, W = image.shape[:2]
        inp = self._preprocess(image)

        output = self.session.run([self.output_name], {self.input_name: inp})[0]
        output = np.squeeze(output)

        if output.shape[0] <= 10:
            output = output.T

        x, y, w, h, conf = output[:, :5].T
        keep = conf >= self.conf_threshold
        x, y, w, h, conf = x[keep], y[keep], w[keep], h[keep], conf[keep]

        scale_x = W / self.input_size
        scale_y = H / self.input_size

        detections = []
        for i in range(len(conf)):
            x1 = int((x[i] - w[i] / 2) * scale_x)
            y1 = int((y[i] - h[i] / 2) * scale_y)
            x2 = int((x[i] + w[i] / 2) * scale_x)
            y2 = int((y[i] + h[i] / 2) * scale_y)

            x1 = max(0, min(W - 1, x1))
            y1 = max(0, min(H - 1, y1))
            x2 = max(0, min(W - 1, x2))
            y2 = max(0, min(H - 1, y2))

            if x2 <= x1 or y2 <= y1:
                continue

            detections.append({
                "bbox": [x1, y1, x2, y2],
                "confidence": float(conf[i]),
                "class": "ffb"
            })

        return detections


FIND + LOAD YOLO MODEL

In [16]:
candidates = glob.glob("**/*.onnx", recursive=True)
print("Found ONNX models:")
for c in candidates:
    print(" -", c)

YOLO_MODEL_PATH = "yolo_model/odroid_h3_deployment/ffb_yolo.onnx"

detector = LocalYOLODetector(
    model_path=YOLO_MODEL_PATH,
    input_size=640,
    conf_threshold=0.25
)

print("[READY] Local YOLO detector initialized")


Found ONNX models:
 - yolo_model\odroid_h3_deployment\ffb_yolo.onnx
[OK] Loaded ONNX YOLO model: yolo_model\odroid_h3_deployment\ffb_yolo.onnx
[READY] Local YOLO detector initialized


RUN YOLO + SAVE CSV & VISUALS

In [17]:
FFB_ID = 17
frames_dir = f"sample_{FFB_ID:03d}"

out_vis_dir = os.path.join(frames_dir, "detections_vis_onnx")
os.makedirs(out_vis_dir, exist_ok=True)

csv_path = os.path.join(frames_dir, "detections_onnx.csv")

CONF_THRESHOLD = 0.15
MASK_BOTTOM_RATIO = 0.35

rgb_files = sorted(glob.glob(os.path.join(frames_dir, "rgb_*.png")))

with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["frame", "x1", "y1", "x2", "y2", "conf", "class"])

    for rf in rgb_files:
        base = os.path.basename(rf)
        try:
            img = cv2.imread(rf)
            H, W = img.shape[:2]

            masked = img.copy()
            y_cut = int(H * (1.0 - MASK_BOTTOM_RATIO))
            masked[y_cut:H, :] = 0

            detections = detector.detect(masked)
            preds = [d for d in detections if d["confidence"] >= CONF_THRESHOLD]

            if not preds:
                writer.writerow([base, "", "", "", "", "", ""])
                continue

            cx0, cy0 = W / 2, H / 2

            def dist2(p):
                x1, y1, x2, y2 = p["bbox"]
                return ((x1+x2)/2-cx0)**2 + ((y1+y2)/2-cy0)**2

            best = min(preds, key=lambda p: (dist2(p), -p["confidence"]))

            x1, y1, x2, y2 = map(int, best["bbox"])
            conf = best["confidence"]

            vis = img.copy()
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(vis, f"ffb {conf:.2f}", (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

            cv2.imwrite(os.path.join(out_vis_dir, base), vis)
            writer.writerow([base, x1, y1, x2, y2, conf, "ffb"])

        except Exception as e:
            print("ERROR:", e)


DEPTH MASK GENERATION

In [20]:
FFB_ID = 17
frames_dir = f"sample_{FFB_ID:03d}"

det_csv   = os.path.join(frames_dir, "detections_rf.csv")
intr_path = os.path.join(frames_dir, "intrinsics.json")

out_mask_dir = os.path.join(frames_dir, "masks")
out_vis_dir  = os.path.join(frames_dir, "masks_vis")
os.makedirs(out_mask_dir, exist_ok=True)
os.makedirs(out_vis_dir,  exist_ok=True)

meta = json.load(open(intr_path))
scale = float(meta["depth_scale"])

SHIRT_Y_RATIO = 0.45

def is_operator_bbox(x1, y1, x2, y2, img_h):
    """
    Returns True if bbox overlaps the operator region.
    """
    return y2 > img_h * (1.0 - SHIRT_Y_RATIO)

def depth_band_mask(depth_u16, bbox, scale, band_cm=10):
    x1, y1, x2, y2 = map(int, bbox)
    h, w = depth_u16.shape

    x1 = max(0, min(w - 1, x1))
    x2 = max(0, min(w - 1, x2))
    y1 = max(0, min(h - 1, y1))
    y2 = max(0, min(h - 1, y2))

    if x2 <= x1 or y2 <= y1:
        return np.zeros_like(depth_u16, np.uint8)

    crop = depth_u16[y1:y2, x1:x2]
    z = crop[crop > 0]
    if z.size == 0:
        return np.zeros_like(depth_u16, np.uint8)

    z_med = np.median(z)
    band = int((band_cm / 100.0) / scale)

    m_roi = (
        (depth_u16 >= (z_med - band)) &
        (depth_u16 <= (z_med + band))
    ).astype(np.uint8) * 255

    mask = np.zeros_like(depth_u16, np.uint8)
    mask[y1:y2, x1:x2] = m_roi[y1:y2, x1:x2]
    return mask

depth_files = {
    os.path.basename(p): p
    for p in glob.glob(os.path.join(frames_dir, "depth_*.png"))
}

def rgb_to_depth(rgb_name):
    return depth_files.get(rgb_name.replace("rgb_", "depth_"))

with open(det_csv, newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        name = row["frame"]

        if not row["x1"]:
            continue

        x1, y1, x2, y2 = map(float, [row["x1"], row["y1"], row["x2"], row["y2"]])

        rgb_path   = os.path.join(frames_dir, name)
        depth_path = rgb_to_depth(name)

        if not depth_path or not os.path.exists(depth_path):
            continue

        rgb = cv2.imread(rgb_path)
        H, W = rgb.shape[:2]

        if is_operator_bbox(x1, y1, x2, y2, H):
            continue

        D = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
        mask = depth_band_mask(D, (x1, y1, x2, y2), scale, band_cm=10)

        mask_path = os.path.join(out_mask_dir, name.replace("rgb_", "mask_"))
        cv2.imwrite(mask_path, mask)

        overlay = rgb.copy()
        overlay[mask == 0] = (overlay[mask == 0] * 0.25).astype(np.uint8)

        vis_path = os.path.join(out_vis_dir, name)
        cv2.imwrite(vis_path, overlay)

print("✅ Masks generated")


✅ Masks generated
